# 🤰 تدريب نموذج الحمل (Pregnancy Model Training)
## XGBoost + Ensemble مع 18 ميزة
### البيانات: Maternal Health Risk Dataset (1014 عينة)

## 1️⃣ استيراد المكتبات

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import xgboost as xgb
import lightgbm as lgb
import shap
import warnings
warnings.filterwarnings('ignore')

print("✅ تم استيراد المكتبات")

## 2️⃣ تحميل البيانات

In [ ]:
base_path = Path.cwd()
project_path = base_path / "RIVA-Offline-AI-Driven-Health-Systems-Resilience"
if not project_path.exists():
    project_path = Path("/content/RIVA-Offline-AI-Driven-Health-Systems-Resilience")

data_path = project_path / "data-storage" / "raw" / "uci_maternal" / "maternal_health_clean.csv"
if not data_path.exists():
    data_path = Path("/content/drive/MyDrive/RIVA-Maternal/maternal_health_clean.csv")

df = pd.read_csv(data_path)
print(f"✅ تم تحميل {len(df)} عينة")

## 3️⃣ تجهيز الميزات (18 ميزة)

In [ ]:
# الميزات الأساسية
X = df[['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate']].copy()
y = df['RiskLevel_encoded'].copy()

# الميزات المشتقة (7)
X['PulsePressure'] = X['SystolicBP'] - X['DiastolicBP']
X['BP_ratio'] = (X['SystolicBP'] / X['DiastolicBP']).round(2)
X['Temp_Fever'] = (X['BodyTemp'] > 37.5).astype(int)
X['HighBP'] = (X['SystolicBP'] > 140).astype(int)
X['MeanBP'] = (X['SystolicBP'] + 2 * X['DiastolicBP']) / 3
X['AgeRisk'] = (X['Age'] > 35).astype(int)
X['HighSugar'] = (X['BS'] > 7).astype(int)

# الميزات التفاعلية (5)
X['BP_BS_Interaction'] = X['SystolicBP'] * X['BS'] / 100
X['Total_Risk_Score'] = X['HighBP'] + X['AgeRisk'] + X['HighSugar'] + X['Temp_Fever']
X['Age_BP_Interaction'] = X['Age'] * X['SystolicBP'] / 100

X['BP_Severity'] = np.where(
    X['SystolicBP'] > 160, 3,
    np.where(X['SystolicBP'] > 140, 2,
    np.where(X['SystolicBP'] > 130, 1, 0))
)

X['BS_Severity'] = np.where(
    X['BS'] > 11, 3,
    np.where(X['BS'] > 8, 2,
    np.where(X['BS'] > 6.5, 1, 0))
)

print(f"✅ تم تجهيز {X.shape[1]} ميزة")
print(f"📋 الميزات: {list(X.columns)}")

## 4️⃣ تدريب النموذج

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# XGBoost مع GridSearch
xgb_model = xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss')

param_grid = {
    'n_estimators': [200, 300, 400],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.15]
}

grid = GridSearchCV(xgb_model, param_grid, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print(f"✅ أفضل باراميترات: {grid.best_params_}")
print(f"✅ أفضل F1-Macro: {grid.best_score_:.4f}")

## 5️⃣ Ensemble (Voting Classifier)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced', random_state=42)
lgbm_model = lgb.LGBMClassifier(n_estimators=300, max_depth=6, class_weight='balanced', random_state=42, verbose=-1)

voting_clf = VotingClassifier(
    estimators=[
        ('xgb', grid.best_estimator_),
        ('rf', rf_model),
        ('lgbm', lgbm_model)
    ],
    voting='soft',
    weights=[2, 1.5, 1.5]
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', voting_clf)
])

pipeline.fit(X_train, y_train)
print("✅ تم تدريب Ensemble")

## 6️⃣ تقييم النموذج

In [ ]:
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['low risk', 'mid risk', 'high risk']))

# مصفوفة الارتباك
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['low', 'mid', 'high'],
            yticklabels=['low', 'mid', 'high'])
plt.title('Confusion Matrix')
plt.show()

## 7️⃣ أهمية الميزات (Feature Importance)

In [ ]:
xgb_best = pipeline.named_steps['classifier'].named_estimators_['xgb']
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': xgb_best.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(data=importance_df.head(10), x='importance', y='feature')
plt.title('Top 10 Feature Importance')
plt.show()

importance_df.head(10)

## 8️⃣ حفظ النموذج

In [ ]:
version = datetime.now().strftime('%Y%m%d_%H%M%S')
model_path = f'/content/drive/MyDrive/RIVA-Maternal/models/pregnancy_model_{version}.pkl'
joblib.dump(pipeline, model_path)
print(f"✅ تم حفظ النموذج في: {model_path}")